# Marker: PDF → marker.json

Runs Marker 2 on one paper and saves the JSON (layout, sections, equations as LaTeX) to Drive.
Nothing else in the extractor runs here; everything after this runs locally from `marker.json`.

**Runtime:** GPU (T4). Run cells 1–5 top to bottom.

Marker's equation reader (surya) needs an inference server. We use a `llama-server` binary built once with CUDA and cached on Drive at `thesis/llamacpp_cuda/`. If that folder is ever lost, run the **Appendix** cell once, then continue.

## 1. Settings

In [ ]:
PAPER_ID   = 'kontoravdi_2010'                       # becomes papers/<PAPER_ID>/ locally
PDF        = '/content/drive/MyDrive/thesis/papers/kontoravdi_2010.pdf'
PAGE_RANGE = None                                     # e.g. '6' or '4-6' (0-indexed). None = whole paper

THESIS     = '/content/drive/MyDrive/thesis'
LLAMA_DIR  = f'{THESIS}/llamacpp_cuda'               # cached llama-server build
OUT_DIR    = f'{THESIS}/papers/{PAPER_ID}'           # marker.json lands here

## 2. Mount Drive and install Marker

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "marker-pdf>=2"

## 3. Point surya at the cached llama-server

The binary was built against CUDA 12, Colab now ships CUDA 13, so the CUDA 12 runtime and cuBLAS that come with the pip `nvidia-*` packages are put on the library path. Paths are found, not hardcoded, so a Python version bump on Colab doesn't break this.

In [ ]:
import os, glob

!mkdir -p /content/llamacpp
!cp -r {LLAMA_DIR}/. /content/llamacpp/
!chmod +x /content/llamacpp/llama-server

cuda12_libs = [d for d in glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/*/lib')
               if d.split('/')[-2] in ('cuda_runtime', 'cublas')]
assert cuda12_libs, 'CUDA 12 runtime libs not found: check the nvidia pip packages'

os.environ['LD_LIBRARY_PATH'] = ':'.join(['/content/llamacpp', *cuda12_libs,
                                          os.environ.get('LD_LIBRARY_PATH', '')])
os.environ['LLAMA_CPP_BINARY']                = '/content/llamacpp/llama-server'
os.environ['SURYA_INFERENCE_BACKEND']         = 'llamacpp'   # otherwise surya tries vLLM via Docker
os.environ['SURYA_INFERENCE_STARTUP_TIMEOUT'] = '600'
os.environ['HF_HOME']                         = f'{THESIS}/hf'  # model weights cached on Drive

!/content/llamacpp/llama-server --version

## 4. Run Marker

In [ ]:
import pathlib
stem = pathlib.Path(PDF).stem
pages = f'--page_range {PAGE_RANGE}' if PAGE_RANGE else ''

!marker_single "{PDF}" --output_format json --output_dir /content/out {pages}
# 'Force-killed llamacpp' at the end is normal: Marker shuts the server down.

## 5. Check equations and save to Drive

In [ ]:
import json, re, shutil, collections

src = f'/content/out/{stem}/{stem}.json'
doc = json.load(open(src))

def walk(b):
    yield b
    for c in b.get('children') or []:
        yield from walk(c)

blocks = list(walk(doc))
eqs = [b for b in blocks if b.get('block_type') == 'Equation']
print(collections.Counter(b.get('block_type') for b in blocks), '\n')
print(len(eqs), 'equations')
for i, e in enumerate(eqs, 1):
    print(f'{i:>3}  ', re.sub(r'<[^>]+>', '', e.get('html') or '').strip()[:160])

os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(src, f'{OUT_DIR}/marker.json')
print('\nsaved ->', f'{OUT_DIR}/marker.json')

In [ ]:
# Optional: download instead of syncing from Drive
from google.colab import files
files.download(f'{OUT_DIR}/marker.json')

---
## Appendix: rebuild llama-server (only if `thesis/llamacpp_cuda/` is missing or step 3 fails to start it)

Takes ~10–15 min on a T4. Saves the build to Drive so step 3 works in future sessions.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j$(nproc) --target llama-server

!mkdir -p {LLAMA_DIR}
!cp -r /content/llama.cpp/build/bin/. {LLAMA_DIR}/
!nvidia-smi --query-gpu=name --format=csv,noheader > {LLAMA_DIR}/BUILT_ON.txt
!cd /content/llama.cpp && git rev-parse --short HEAD >> {LLAMA_DIR}/BUILT_ON.txt
!ls {LLAMA_DIR}